<a href="https://colab.research.google.com/github/shaifaliverma99/App-Landing-Page/blob/main/eeg_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Unzip the Project
Let's extract the contents of the zip file into the current directory.

In [ ]:
import zipfile
import os

zip_path = '/content/NeuroTrans-EEG-Seizure.zip'
extract_path = '/content/NeuroTrans-EEG-Seizure'

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print(f"Extracted all files to: {extract_path}")
print(os.listdir(extract_path))

Extracted all files to: /content/NeuroTrans-EEG-Seizure
['NeuroTrans-EEG-Seizure']


### Mount Google Drive
Use this cell to mount your Google Drive to access your datasets.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### Inspect Project Structure
Let's see what files are inside the extracted repository so we know how to run it.

In [ ]:
import os

project_inner_dir = '/content/NeuroTrans-EEG-Seizure/NeuroTrans-EEG-Seizure'
if os.path.exists(project_inner_dir):
    print("Files in project:", os.listdir(project_inner_dir))
else:
    print("Project inner directory not found. Listing root extraction directory:", os.listdir('/content/NeuroTrans-EEG-Seizure'))

Files in project: ['tests', 'run_experiments.py', 'configs', '.gitignore', 'README.md', 'make_synthetic_data.py', 'docs', 'scripts', 'explain_report.py', 'benchmark_speed.py', 'prepare_data.py', 'original_repo', 'make_report.py', 'requirements.txt', 'notebooks', 'seizure']


### Step 1: Install Requirements
Let's install the project dependencies specified in `requirements.txt`.

In [ ]:
!pip install -r /content/NeuroTrans-EEG-Seizure/NeuroTrans-EEG-Seizure/requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 9.7 MB/s  0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.7/7.7 MB 30.3 MB/s  0:00:00
  Created wheel for pyedflib: filename=pyedflib-0.1.42-cp313-cp313-linux_x86_64.whl size=2852725 sha256=28297bfdc98c89e78aac48717d488b4c01aec48c32aa654ed060a436ae6dd074
  Stored in directory: /root/.cache/pip/wheels/ac/16/3e/5299c8a6b023bfd3d7204240ca5de940a49bede1edb30015ab
Successfully built pyedflib
  Attempting uninstall: decorator
    Found existing installation: decorator 4.4.2
    Uninstalling decorator-4.4.2:
      Successfully uninstalled decorator-4.4.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [mne]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requ

### Step 2: Check Usage for Data Preparation and Experiments
Let's run `prepare_data.py` and `run_experiments.py` with the `--help` flag to see how they should be configured to use your Google Drive dataset paths.

In [ ]:
%cd /content/NeuroTrans-EEG-Seizure/NeuroTrans-EEG-Seizure
!python prepare_data.py --help
print("-"*50)
!python run_experiments.py --help

/content/NeuroTrans-EEG-Seizure/NeuroTrans-EEG-Seizure
usage: prepare_data.py [-h] [--config CONFIG] [--set SET]
                       [--patients [PATIENTS ...]]

options:
  -h, --help            show this help message and exit
  --config CONFIG
  --set SET             override, e.g. data.stride_sec=2.0
  --patients [PATIENTS ...]
--------------------------------------------------
^C


### Step 3: Inspect Configuration Files
Let's check the configurations inside the `configs/` folder to understand where the dataset path needs to be updated.

In [ ]:
import os

configs_dir = '/content/NeuroTrans-EEG-Seizure/NeuroTrans-EEG-Seizure/configs'
if os.path.exists(configs_dir):
    config_files = os.listdir(configs_dir)
    print("Configuration files:", config_files)

    # Display content of the first config file if it exists
    if config_files:
        first_config = os.path.join(configs_dir, config_files[0])
        print(f"\n--- Content of {config_files[0]} ---")
        with open(first_config, 'r') as f:
            print(f.read())
else:
    print("Configs directory not found.")

Configuration files: ['colab.yaml', 'default.yaml', 'smoke_test.yaml', 'b200.yaml']

--- Content of colab.yaml ---
# Free / Pro Colab (T4, L4, A100). Few CPU cores -> few workers.
_base_: default.yaml
data:
  n_jobs: 2
train:
  num_workers: 2
  batch_size: 128
  eval_batch_size: 256
  epochs: 20



### Step 4: Inspect `default.yaml` for Dataset Paths
Let's print out the contents of the base configuration file `default.yaml` to find where the paths are defined.

In [ ]:
default_config_path = '/content/NeuroTrans-EEG-Seizure/NeuroTrans-EEG-Seizure/configs/default.yaml'
with open(default_config_path, 'r') as f:
    print(f.read())

# ---------------------------------------------------------------------------
# PIDF-Net default configuration.  Override anything from the command line:
#   python run_lopo.py --set train.epochs=5 --set data.data_dir=/content/drive/MyDrive/chbmit
# ---------------------------------------------------------------------------
data:
  data_dir: ./chbmit_data        # folder that contains chb01/, chb02/, ... (each with *.edf + chbXX-summary.txt)
  cache_dir: ./cache             # preprocessed windows are stored here (safe to delete)
  patients: all                  # "all" = every chbXX folder found, or a list: [chb01, chb02, ...]
  fs: 256                        # CHB-MIT sampling rate (files are resampled if different)
  window_sec: 4.0
  stride_sec: 4.0                # 4.0 = non-overlapping; 2.0 = 50% overlap (2x storage)
  bandpass: [0.5, 40.0]          # Hz, zero-phase Butterworth
  normalize: robust_file         # robust_file | zscore_file | none  (label-free, per-recording)
  clip:

### Step 5: Run Data Preparation and Experiments with Drive Dataset
Replace `/content/drive/MyDrive/YOUR_DATASET_FOLDER` in the cell below with the actual path to your CHB-MIT dataset on Google Drive.

In [ ]:
# Change directory to project root
%cd /content/NeuroTrans-EEG-Seizure/NeuroTrans-EEG-Seizure

# Define your Google Drive dataset path here
DRIVE_DATASET_PATH = "/content/drive/MyDrive/CHB-MIT"

# 1. Run Data Preparation (Preprocess the EDF files and build caches)
!python prepare_data.py --config configs/colab.yaml --set data.data_dir={DRIVE_DATASET_PATH}

# 2. Run Experiments (Train & Evaluate the model)
# (We can use configs/colab.yaml since we are in Colab)
!python run_experiments.py --config configs/colab.yaml --set data.data_dir={DRIVE_DATASET_PATH}

/content/NeuroTrans-EEG-Seizure/NeuroTrans-EEG-Seizure
Preprocessing 210 EDF files from 24 patients -> cache/1945b5dae8
[Parallel(n_jobs=2)]: Using backend LokyBackend with 2 concurrent workers.
[Parallel(n_jobs=2)]: Done   1 tasks      | elapsed:    8.3s
[Parallel(n_jobs=2)]: Done   4 tasks      | elapsed:   13.7s
[Parallel(n_jobs=2)]: Done   9 tasks      | elapsed:   28.8s
[Parallel(n_jobs=2)]: Done  14 tasks      | elapsed:   45.3s
[Parallel(n_jobs=2)]: Done  21 tasks      | elapsed:  1.1min
[Parallel(n_jobs=2)]: Done  28 tasks      | elapsed:  1.7min
[Parallel(n_jobs=2)]: Done  37 tasks      | elapsed:  2.5min
[Parallel(n_jobs=2)]: Done  46 tasks      | elapsed:  3.8min
[Parallel(n_jobs=2)]: Done  57 tasks      | elapsed:  5.5min
[Parallel(n_jobs=2)]: Done  68 tasks      | elapsed:  6.5min
[Parallel(n_jobs=2)]: Done  81 tasks      | elapsed:  7.8min
[Parallel(n_jobs=2)]: Done  94 tasks      | elapsed:  8.3min
[Parallel(n_jobs=2)]: Done 109 tasks      | elapsed:  9.1min
[Parallel(n_

### Step 6: Evaluate Model Performance & Generate Reports

When `run_experiments.py` runs, it trains the model and automatically performs evaluation according to the protocol defined in your config (by default, Leave-One-Patient-Out cross-validation).

The training and evaluation process will automatically save metrics, checkpoints, and predictions inside a directory (usually named `outputs/` or `results/`).

You can summarize these results, compute overall metrics (like Sensitivity, False Alarm Rate per Hour, and Latency), and compile tables/figures by running the project's dedicated evaluation scripts:

* **`make_report.py`**: Compiles the cross-validation results into a clean performance report.
* **`explain_report.py`**: Can be used to run post-hoc explanations or deeper metric breakdowns.

In [ ]:
import os

def find_file(filename, search_path):
    for root, dirs, files in os.walk(search_path):
        if filename in files:
            return root
    return None

target_dir = find_file("make_report.py", "/content")

if target_dir:
    print(f"Found project directory at: {target_dir}")
    %cd {target_dir}
    !python make_report.py
else:
    print("Could not find make_report.py anywhere inside /content.")

Could not find make_report.py anywhere inside /content.


In [ ]:
import os
import shutil
import zipfile
from google.colab import drive

# 1. Ensure Google Drive is mounted
if not os.path.exists('/content/drive/MyDrive'):
    print("Mounting Google Drive...")
    drive.mount('/content/drive')

# Direct path from your Google Drive list
drive_zip_path = '/content/drive/MyDrive/NEURO_FUSION_project.zip'
local_zip_path = '/content/NeuroTrans-EEG-Seizure.zip'
extract_path = '/content/NeuroTrans-EEG-Seizure'

# 2. Copy file locally if it exists on Drive
if os.path.exists(drive_zip_path):
    print(f"Found project zip on Google Drive at: {drive_zip_path}")
    print("Copying to local runtime...")
    shutil.copy(drive_zip_path, local_zip_path)
else:
    print(f"Could not find the zip at {drive_zip_path}. Checking all zip files in Drive...")
    for item in os.listdir('/content/drive/MyDrive'):
        if item.endswith('.zip') and 'FUSION' in item:
            found_path = os.path.join('/content/drive/MyDrive', item)
            print(f"Found match: {found_path}. Copying...")
            shutil.copy(found_path, local_zip_path)
            break

# 3. Extracting the file
if os.path.exists(local_zip_path):
    print(f"Extracting {local_zip_path} to {extract_path}...")
    with zipfile.ZipFile(local_zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path)
    print("Extraction complete!")
    print("Contents of extracted root:", os.listdir(extract_path))
else:
    print("Failed to find or copy the project zip file.")

Mounting Google Drive...
Mounted at /content/drive
Found project zip on Google Drive at: /content/drive/MyDrive/NEURO_FUSION_project.zip
Copying to local runtime...
Extracting /content/NeuroTrans-EEG-Seizure.zip to /content/NeuroTrans-EEG-Seizure...
Extraction complete!
Contents of extracted root: ['LICENSE.txt', 'artifacts', 'docs', 'configs', 'requirements-dgx-b200.txt', 'data', '.gitignore', 'requirements-colab.txt', 'src', 'notebooks', 'README.md', 'scripts', 'tests', 'reports']


In [ ]:
import os

project_root = '/content/NeuroTrans-EEG-Seizure'
print("Contents of root:", os.listdir(project_root))

# Let's inspect subfolders to locate our python scripts
for folder in ['src', 'scripts']:
    folder_path = os.path.join(project_root, folder)
    if os.path.exists(folder_path):
        print(f"\nContents of {folder}:", os.listdir(folder_path))

Contents of root: ['LICENSE.txt', 'artifacts', 'docs', 'configs', 'requirements-dgx-b200.txt', 'data', '.gitignore', 'requirements-colab.txt', 'src', 'notebooks', 'README.md', 'scripts', 'tests', 'reports']

Contents of src: ['training', 'models', '__init__.py', 'data', 'evaluation', 'utils.py']

Contents of scripts: ['run_all_checks.sh', 'train_alzheimer.py', '__init__.py', 'build_manifests.py', 'audit_manifests.py', 'diagnose_gpu.py', 'run_research_audit.py', 'train_parkinson_drawings.py', 'train_epilepsy.py', 'train_parkinson_voice.py', 'check_fusion.py']


In [ ]:
!pip install -r /content/NeuroTrans-EEG-Seizure/requirements-colab.txt

# Let's inspect scripts/train_epilepsy.py or run the training
import os
if os.path.exists('/content/NeuroTrans-EEG-Seizure/scripts/train_epilepsy.py'):
    print("Found training script. Ready to run.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 66.1 MB/s  0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.3/17.3 MB 78.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 72.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 950.3/950.3 kB 38.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9/9 [jupyterlab]
Found training script. Ready to run.


In [1]:
import subprocess
import os

project_root = '/content/NeuroTrans-EEG-Seizure'
if os.path.exists(project_root):
    # Check if a .git folder exists or retrieve remote origin using git
    try:
        output = subprocess.check_output(
            ["git", "remote", "-v"],
            cwd=project_root,
            stderr=subprocess.STDOUT
        ).decode()
        print("Git Remotes:")
        print(output)
    except Exception as e:
        print("Could not retrieve git remotes directly. Let's check for any hidden .git files/directories:")
        all_files = os.listdir(project_root)
        print("Root contents:", all_files)
        git_dir = os.path.join(project_root, '.git')
        if os.path.exists(git_dir):
            print(".git directory exists! Checking config file:")
            config_path = os.path.join(git_dir, 'config')
            if os.path.exists(config_path):
                with open(config_path, 'r') as f:
                    print(f.read())
            else:
                print(".git/config not found.")
        else:
            print("This extracted directory is not initialized as a git repository locally.")
else:
    print("Project root path does not exist.")

Project root path does not exist.


In [2]:
import os

possible_readme_paths = [
    '/content/drive/MyDrive/NEURO_FUSION/README.md',
    '/content/drive/MyDrive/NEURO_FUSION_COLAB/README.md'
]

found_any = False
for path in possible_readme_paths:
    if os.path.exists(path):
        print(f"=== Found README at {path} ===\n")
        with open(path, 'r') as f:
            content = f.read()
            # Scan for github links
            lines = content.split('\n')
            for line in lines:
                if 'github.com' in line:
                    print(line)
        found_any = True

if not found_any:
    print("Could not find direct README.md in Google Drive folders.")

Could not find direct README.md in Google Drive folders.


In [3]:
import os

readme_path = '/content/NeuroTrans-EEG-Seizure/README.md'
if os.path.exists(readme_path):
    print("=== Found README.md inside extracted project ===")
    with open(readme_path, 'r') as f:
        for line in f:
            if 'github.com' in line:
                print(line.strip())
else:
    print("README.md not found at standard root. Searching subdirectories...")
    for root, dirs, files in os.walk('/content/NeuroTrans-EEG-Seizure'):
        if 'README.md' in files:
            sub_readme = os.path.join(root, 'README.md')
            print(f"\n=== Found README.md at {sub_readme} ===")
            with open(sub_readme, 'r') as f:
                for line in f:
                    if 'github.com' in line:
                        print(line.strip())

README.md not found at standard root. Searching subdirectories...


In [ ]:
import os

def search_drive_for_csvs(root_path):
    found_csvs = []
    for root, dirs, files in os.walk(root_path):
        for file in files:
            if file.endswith('.csv') or 'epilepsy' in file.lower():
                found_csvs.append(os.path.join(root, file))
    return found_csvs

print("Searching for epilepsy files or any CSVs on Google Drive...")
results = search_drive_for_csvs('/content/drive/MyDrive')
for path in results[:30]:
    print(f"- {path}")

Searching for epilepsy files or any CSVs on Google Drive...
- /content/drive/MyDrive/epilepsy nb3.ipynb
- /content/drive/MyDrive/epilepsy.gprj
- /content/drive/MyDrive/Colab Notebooks/epilepsy.ipynb
- /content/drive/MyDrive/Colab Notebooks/epilepsy_colab.ipynb
- /content/drive/MyDrive/Datasets/Epilepsy.zip
- /content/drive/MyDrive/NEURO_FUSION/datasets/Epilepsy/Epileptic_Seizure_Recognition.csv
- /content/drive/MyDrive/NeuroDetect_Epilepsy_Colab_Full_Project/NeuroDetect_Epilepsy_Deadline_Package/notebooks/NeuroDetect_Epilepsy_Colab.ipynb


### Locate Dataset in Google Drive
Let's search and list the top-level directories and contents of your Google Drive to find the exact dataset folder name.

In [ ]:
import os

drive_path = '/content/drive/MyDrive/'

if os.path.exists(drive_path):
    print("Listing top-level directories in Google Drive:")
    contents = os.listdir(drive_path)
    for item in sorted(contents):
        full_path = os.path.join(drive_path, item)
        if os.path.isdir(full_path):
            print(f"  [Directory] {item}")
        else:
            print(f"  [File]      {item}")
else:
    print("Google Drive is not mounted or path not found. Please ensure the Mount Google Drive step ran successfully.")

Listing top-level directories in Google Drive:
  [File]      00_neuro_multimodel_framework_GPU.ipynb.gdoc
  [File]      10th_.pdf
  [File]      12th.pdf
  [File]      2026-06-14 10-07-27 (1).pdf
  [File]      2026-06-14 10-07-27.pdf
  [File]      AFCAT.pdf
  [File]      Apprentice_Letter-Shaifali Verma-2349335.pdf
  [Directory] CHB-MIT
  [Directory] Colab Notebooks
  [File]      Complaint Submitted - Internet Crime Complaint Center (IC3).pdf
  [Directory] Datasets
  [File]      Datasets.zip
  [File]      Edusat Registrations.pdf
  [File]      IAF : After Payment Preview.pdf
  [File]      IMG_20240106_221245.jpg
  [File]      IMG_20240119_182959.jpg
  [File]      JD for Associate_Developer -Technology Engineer careerzodiac.docx
  [File]      JD for Associate_Developer -Technology Engineer careerzodiac.gdoc
  [File]      LCAT_Certificate_Shaifali_Verma.pdf
  [File]      LOR1.pdf
  [File]      LOR2.pdf
  [File]      Literature Survey:.gdoc
  [Directory] NEURO_FUSION
  [Directory] NEURO_FU